# 04 — Preprocessing y headways  (auto-generado por build_notebook_04.py)

Este notebook aplica el pipeline de Fase 2 al dataset `clean_gps.parquet` para
producir `cleaned_gps_E{empresa}.parquet` y `headways_E{empresa}.parquet` por
corredor. Formulación adoptada: **Opción C.2 — trailing crossing** (ver
`docs/decisiones-headway-fase2.md §2`).

Parámetros productivos congelados en `config.py` desde
`docs/decisiones-headway-fase2.md §3`.

In [ ]:

import polars as pl
import numpy as np
from pathlib import Path
import os

# Locate clean_gps.parquet under /kaggle/input (or local working directory).
candidates = list(Path("/kaggle/input").rglob("clean_gps.parquet")) if Path("/kaggle/input").exists() else []
if not candidates:
    candidates = list(Path(".").rglob("clean_gps.parquet"))
if not candidates:
    raise FileNotFoundError("clean_gps.parquet not found. Expected at /kaggle/input/**/clean_gps.parquet")
INPUT = candidates[0]
print(f"Input: {INPUT}")

OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
OUTPUT_DIR.mkdir(exist_ok=True)
print(f"Output dir: {OUTPUT_DIR}")

EMPRESAS = [2, 59]

## Module: config

Parámetros productivos congelados desde `docs/decisiones-headway-fase2.md §3`.
Cualquier cambio requiere actualizar ese documento primero.

In [ ]:
"""Configuration and frozen parameters for the Fase 2 preprocessing pipeline.

All productive parameter values are locked to docs/decisiones-headway-fase2.md §3.
Changing any value requires updating that document first (versioned decision),
then updating the literal here. The freeze-assertion test in
tests/preprocessing/test_config.py encodes this contract as executable checks.
"""
import math
from dataclasses import dataclass
from typing import Mapping

# ---------------------------------------------------------------------------
# Coordinate constants — local flat-Earth at Arequipa (-16.4°)
# ---------------------------------------------------------------------------

LAT_DEG_M: float = 111_000.0
LON_DEG_M: float = 111_000.0 * math.cos(math.radians(-16.4))

# ---------------------------------------------------------------------------
# Quality thresholds (decisiones-limpieza-fase2 §2 rows 4-5)
# ---------------------------------------------------------------------------

MAX_PLAUSIBLE_SPEED_KMH: float = 80.0
MAX_PLAUSIBLE_JUMP_M: float = 500.0

# ---------------------------------------------------------------------------
# Trip segmentation (decisión §3.3 of decisiones-limpieza-fase2)
# ---------------------------------------------------------------------------

GAP_CUT_SECONDS: int = 30 * 60         # 30-minute gap between consecutive pings
TERMINAL_BAND_M: float = 200.0         # within X m of s_min / s_max → terminal candidate
TERMINAL_DWELL_SECONDS: int = 5 * 60   # stopped > 5 min near a terminal → cut
TERMINAL_MAX_SPEED_KMH: float = 5.0    # stopped threshold for terminal-dwell detection


# ---------------------------------------------------------------------------
# Frozen productive parameters
# ---------------------------------------------------------------------------

@dataclass(frozen=True)
class ProductiveParams:
    """Frozen contract — every field mirrors docs/decisiones-headway-fase2.md §3.

    Changing a value requires updating that document FIRST (versioned), then
    this file. The freeze-assertion test in test_config.py turns this into
    executable code.
    """

    grid_seconds: int = 60
    min_speed_for_centerline_kmh: float = 10.0
    centerline_latlon_quantile_lo: float = 0.005
    centerline_latlon_quantile_hi: float = 0.995
    centerline_n_bins: int = 50
    centerline_trim_pct: float = 0.025
    centerline_smooth_win: int = 5
    lateral_offset_threshold_m: float = 300.0
    direction_smooth_win: int = 5
    min_buses_per_snapshot: int = 2
    # Max staleness in minutes for a historical crossing to count as a real
    # trailing pair. Older crossings → emit delta_t_min = NULL. Bound exists
    # because multi-filar corridors (e.g. E2 in Arequipa) project unrelated
    # buses to the same s; without this bound, np.searchsorted finds ancient
    # crossings and reports them as valid headways. See decisiones-headway-fase2 §3.
    max_interpolation_lookback_minutes: float = 30.0
    # Lateral distance threshold (meters) between bus_front and bus_back to
    # consider them on the same track. Pairs with |lateral_m_front -
    # lateral_m_back| > threshold are filtered out as cross-street pairs.
    # Per-empresa override available via EmpresaConfig.lateral_pair_threshold_m_override.
    # See decisiones-headway-fase2 §3 (multi-filar-disambiguation).
    lateral_pair_threshold_m: float = 50.0


PRODUCTIVE_PARAMS = ProductiveParams()


# ---------------------------------------------------------------------------
# Per-empresa configuration
# ---------------------------------------------------------------------------

@dataclass(frozen=True)
class EmpresaConfig:
    """Per-empresa settings.

    has_heading: E2/E4 report a `direccion` field usable as cross-check;
                 E58/E59 do not.
    centerline_sample_cap: maximum pings used to build the centerline.
    lateral_offset_threshold_m_override: when set, overrides
        PRODUCTIVE_PARAMS.lateral_offset_threshold_m for this empresa.
        Used for Caveat 3 monitoring (see decisiones-headway-fase2 §4).
    """

    empresaid: int
    has_heading: bool
    centerline_sample_cap: int = 50_000
    lateral_offset_threshold_m_override: float | None = None
    # Per-empresa override for the lateral pair filter threshold (meters).
    # When set, overrides PRODUCTIVE_PARAMS.lateral_pair_threshold_m for this
    # empresa. Used after Kaggle calibration of the |lateral_delta| histogram.
    lateral_pair_threshold_m_override: float | None = None


EMPRESA_CONFIG: Mapping[int, EmpresaConfig] = {
    2:  EmpresaConfig(empresaid=2,  has_heading=True),
    59: EmpresaConfig(empresaid=59, has_heading=False),
}


def lateral_threshold_for(empresaid: int) -> float:
    """Return the effective lateral offset threshold for a given empresa.

    Checks EmpresaConfig.lateral_offset_threshold_m_override first; falls
    back to PRODUCTIVE_PARAMS.lateral_offset_threshold_m (Caveat 3 hook).
    """
    cfg = EMPRESA_CONFIG[empresaid]
    if cfg.lateral_offset_threshold_m_override is not None:
        return cfg.lateral_offset_threshold_m_override
    return PRODUCTIVE_PARAMS.lateral_offset_threshold_m


def lateral_pair_threshold_for(empresaid: int) -> float:
    """Return the effective lateral pair filter threshold for a given empresa.

    Checks EmpresaConfig.lateral_pair_threshold_m_override first; falls back
    to PRODUCTIVE_PARAMS.lateral_pair_threshold_m. Returns the global default
    for empresas not in EMPRESA_CONFIG (graceful missing-key handling).

    Used by compute_pairs to decide which (front, back) pairs are cross-street
    contamination and should be filtered out.
    """
    cfg = EMPRESA_CONFIG.get(empresaid)
    if cfg is not None and cfg.lateral_pair_threshold_m_override is not None:
        return cfg.lateral_pair_threshold_m_override
    return PRODUCTIVE_PARAMS.lateral_pair_threshold_m

## Module: corridor

Construcción del trazado del corredor via PCA + binned median.
Fuente: `build_notebook_03.py` líneas 279-361.

In [ ]:
"""Corridor centerline construction for the preprocessing pipeline.

Extracts an ordered (n_bins, 2) lat/lon polyline from GPS pings via:
  1. Geographic-outlier filter (IQR box trim at configurable quantiles).
  2. PCA to find the principal axis of the corridor.
  3. Binned median along the principal axis.
  4. Smoothing of the secondary (cross-corridor) coordinate.
  5. Back-transformation to (lat, lon).

Source: derived from build_notebook_03.py lines 279-361.
"""
from __future__ import annotations

import numpy as np
import polars as pl



def _filter_geographic_outliers(
    points_latlon: np.ndarray,
    q: tuple[float, float] = (
        PRODUCTIVE_PARAMS.centerline_latlon_quantile_lo,
        PRODUCTIVE_PARAMS.centerline_latlon_quantile_hi,
    ),
) -> np.ndarray:
    """Trim pings outside the [q_lo, q_hi] quantile box of lat and lon.

    Failure mode: if this filter is broken (too loose or too tight) the PCA
    principal axis tilts off-corridor or the sample becomes too small. The
    test_corridor.py outlier test catches regressions in both directions.

    Args:
        points_latlon: (n, 2) array of (lat, lon) values.
        q: (q_lo, q_hi) quantile tuple, default from PRODUCTIVE_PARAMS.

    Returns:
        Filtered (n_kept, 2) array.
    """
    pts = np.asarray(points_latlon, dtype=float)
    lat_lo, lat_hi = np.quantile(pts[:, 0], q)
    lon_lo, lon_hi = np.quantile(pts[:, 1], q)
    mask = (
        (pts[:, 0] >= lat_lo) & (pts[:, 0] <= lat_hi)
        & (pts[:, 1] >= lon_lo) & (pts[:, 1] <= lon_hi)
    )
    return pts[mask]


def build_centerline(
    gps: pl.DataFrame,
    empresaid: int,
    rng_seed: int = 42,
) -> np.ndarray:
    """Build the ordered (m, 2) lat/lon polyline for one empresa.

    Pipeline: geographic-outlier filter → PCA → binned median → trim → smooth
    → back-transform to (lat, lon).

    Args:
        gps: DataFrame with columns (empresaid, unidadid, lat, lon, speed_kmh).
             speed_kmh must already be populated — call
             projection.attach_observed_speed first.
        empresaid: which empresa to build the centerline for.
        rng_seed: seed for deterministic random sampling when the GPS sample
                  exceeds centerline_sample_cap.

    Returns:
        np.ndarray shape (m, 2) of (lat, lon) ordered along the principal axis,
        where m <= PRODUCTIVE_PARAMS.centerline_n_bins (bins with < 5 samples
        are silently dropped).

    Failure mode: PCA sign flip (centered data → eigenvector pointing west)
    produces a reversed polyline. test_corridor.py checks that the first vertex
    is near LON_START and the last is near LON_END of the synthetic route.
    """
    cfg = EMPRESA_CONFIG[empresaid]
    params = PRODUCTIVE_PARAMS

    moving = (
        gps.filter(
            (pl.col("empresaid") == empresaid)
            & (pl.col("speed_kmh") >= params.min_speed_for_centerline_kmh)
        )
        .select(["lat", "lon"])
    )

    rng = np.random.default_rng(rng_seed)
    sample: np.ndarray = moving.to_numpy()
    if len(sample) > cfg.centerline_sample_cap:
        idx = rng.choice(len(sample), size=cfg.centerline_sample_cap, replace=False)
        sample = sample[idx]

    return _build_centerline_from_points(
        sample,
        n_bins=params.centerline_n_bins,
        trim_pct=params.centerline_trim_pct,
        smooth_win=params.centerline_smooth_win,
    )


def _build_centerline_from_points(
    points_latlon: np.ndarray,
    n_bins: int = PRODUCTIVE_PARAMS.centerline_n_bins,
    trim_pct: float = PRODUCTIVE_PARAMS.centerline_trim_pct,
    smooth_win: int = PRODUCTIVE_PARAMS.centerline_smooth_win,
) -> np.ndarray:
    """Inner implementation of centerline construction from a point array.

    Separated from build_centerline to make the algorithm unit-testable with
    arbitrary point sets (not tied to a polars DataFrame or empresa).

    Args:
        points_latlon: (n, 2) array of (lat, lon) values.
        n_bins: number of bins along the principal axis.
        trim_pct: fraction of extreme principal-axis positions to drop.
        smooth_win: rolling mean window for the cross-corridor coordinate.

    Returns:
        np.ndarray shape (m, 2) of (lat, lon), m <= n_bins.
    """
    pts = _filter_geographic_outliers(points_latlon)

    centroid = pts.mean(axis=0)
    centered = pts - centroid

    # PCA via eigen-decomposition of the 2×2 covariance matrix.
    cov = np.cov(centered.T)
    eigvals, eigvecs = np.linalg.eigh(cov)
    order = np.argsort(eigvals)[::-1]
    eigvecs = eigvecs[:, order]

    projected = centered @ eigvecs     # (n, 2)
    t1 = projected[:, 0]               # principal axis coordinate
    t2 = projected[:, 1]               # cross-corridor coordinate

    # Trim extreme percentiles along the principal axis.
    lo, hi = np.quantile(t1, [trim_pct, 1.0 - trim_pct])
    mask = (t1 >= lo) & (t1 <= hi)
    t1, t2 = t1[mask], t2[mask]

    # Bin along the principal axis; take median cross-corridor coord per bin.
    bins = np.linspace(t1.min(), t1.max(), n_bins + 1)
    bin_idx = np.clip(np.digitize(t1, bins) - 1, 0, n_bins - 1)

    cl_proj: list[list[float]] = []
    for i in range(n_bins):
        m = bin_idx == i
        if m.sum() < 5:
            continue
        cl_proj.append([0.5 * (bins[i] + bins[i + 1]), float(np.median(t2[m]))])

    if not cl_proj:
        raise ValueError(
            f"build_centerline produced no bins with >= 5 points for n_bins={n_bins}. "
            "The GPS sample may be too small or too sparse."
        )

    cl_proj_arr = np.array(cl_proj)

    # Smooth the cross-corridor coordinate with a rolling mean.
    if smooth_win > 1 and len(cl_proj_arr) >= smooth_win:
        kernel = np.ones(smooth_win) / smooth_win
        cl_proj_arr[:, 1] = np.convolve(cl_proj_arr[:, 1], kernel, mode="same")

    # Back-transform from PCA space to (lat, lon).
    cl_latlon: np.ndarray = cl_proj_arr @ eigvecs.T + centroid
    return cl_latlon

## Module: projection

Speed observado (`step_m / dt_s`, no `velocidad`) y proyección arc-length `s`.
Filtra pings off-route con `lateral_m > LATERAL_OFFSET_THRESHOLD_M`.

In [ ]:
"""Speed attachment and arc-length projection for the preprocessing pipeline.

Provides:
  attach_observed_speed — compute step_m, dt_s, speed_kmh per (empresaid, unidadid)
                          and DROP GPS-jump pairs per spec R11 (pair-level discard,
                          not row-level nulling).
  project_to_centerline — project pings onto a polyline, compute s and lateral_m,
                          drop off-route rows.

Source: derived from build_notebook_03.py lines 246-273 (speed) and
        390-468 (projection + off-route filter).
"""
from __future__ import annotations

import numpy as np
import polars as pl



def attach_observed_speed(gps: pl.DataFrame) -> pl.DataFrame:
    """Add columns (lat_prev, lon_prev, time_prev, step_m, dt_s, speed_kmh) by
    diffing successive rows of the same (empresaid, unidadid), then discard
    GPS-jump pairs per spec R11.

    speed_kmh is computed as step_m / dt_s * 3.6 (observed speed from GPS
    displacement). The raw `velocidad` field is intentionally NOT used (spec R11,
    decisiones-limpieza-fase2 §2.3).

    Pair-level discard (spec R11) — rows are DROPPED (not nulled) when:
      1. speed_kmh > MAX_PLAUSIBLE_SPEED_KMH (80 km/h): GPS jump or data error.
      2. step_m > MAX_PLAUSIBLE_JUMP_M (500 m) AND dt_s <= 60 s: implausible jump.

    The first ping per bus has no previous ping, so step_m and dt_s are null
    and speed_kmh is null. These rows are KEPT (null speed is not an outlier —
    it is missing data for the leading ping only). The filter conditions
    explicitly preserve null-speed rows.

    Output frame has fewer rows than input when GPS jumps are present.

    Source: build_notebook_03.py lines 250-272 (extended for R11 pair-level discard).
    """
    gps = gps.with_columns([
        pl.col("lat").shift(1).over(["empresaid", "unidadid"]).alias("lat_prev"),
        pl.col("lon").shift(1).over(["empresaid", "unidadid"]).alias("lon_prev"),
        pl.col("time").shift(1).over(["empresaid", "unidadid"]).alias("time_prev"),
    ])
    gps = gps.with_columns([
        (
            ((pl.col("lat") - pl.col("lat_prev")) * LAT_DEG_M) ** 2
            + ((pl.col("lon") - pl.col("lon_prev")) * LON_DEG_M) ** 2
        ).sqrt().alias("step_m"),
        (pl.col("time") - pl.col("time_prev")).dt.total_seconds().alias("dt_s"),
    ])
    gps = gps.with_columns(
        pl.when(pl.col("dt_s").is_not_null() & (pl.col("dt_s") > 0))
          .then(pl.col("step_m") / pl.col("dt_s") * 3.6)
          .otherwise(None)
          .alias("speed_kmh")
    )
    # Pair-level discard criterion 1 (spec R11): drop rows where speed > 80 km/h.
    # Null speed (first ping per bus) is preserved — it is not a GPS-jump outlier.
    gps = gps.filter(
        pl.col("speed_kmh").is_null() | (pl.col("speed_kmh") <= MAX_PLAUSIBLE_SPEED_KMH)
    )
    # Pair-level discard criterion 2 (spec R11): drop rows where step_m > 500 m
    # AND dt_s <= 60 s. This catches teleporting pings that briefly exceed the
    # jump threshold within a 1-minute window.
    # The first ping per bus has step_m = null (no previous ping) — these must
    # be kept. Polars propagates null through comparisons, so we must explicitly
    # preserve null-step_m rows with step_m.is_null() as an OR guard.
    gps = gps.filter(
        pl.col("step_m").is_null()
        | ~(
            (pl.col("step_m") > MAX_PLAUSIBLE_JUMP_M)
            & (pl.col("dt_s") <= 60)
        )
    )
    return gps


def project_to_centerline(
    gps: pl.DataFrame,
    centerline_latlon: np.ndarray,
    empresaid: int,
    chunk_size: int = 10_000,
) -> pl.DataFrame:
    """Project each ping onto the centerline polyline, compute arc-length s and
    lateral_m, then drop pings where lateral_m > lateral_threshold_for(empresaid).

    Args:
        gps: rows belonging to a SINGLE empresa with columns
             (empresaid, unidadid, time, lat, lon, speed_kmh).
        centerline_latlon: (m, 2) array of (lat, lon) from corridor.build_centerline.
        empresaid: used to look up the lateral offset threshold.
        chunk_size: number of pings to process per numpy batch (bounds peak memory).

    Returns:
        pl.DataFrame with added columns (s: Float64, lateral_m: Float64) after
        applying the lateral off-route filter. Pings with lateral_m above the
        threshold are removed.

    Failure mode: if chunk boundaries produce s discontinuities, monotonicity
    of s for a straight on-route bus breaks. test_projection.py catches this.

    Source: build_notebook_03.py lines 390-468.
    """
    gps_e = gps.filter(pl.col("empresaid") == empresaid)
    if gps_e.is_empty():
        return gps_e.with_columns([
            pl.lit(None, dtype=pl.Float64).alias("s"),
            pl.lit(None, dtype=pl.Float64).alias("lateral_m"),
        ])

    points_latlon = gps_e.select(["lat", "lon"]).to_numpy()
    s_arr, lateral_arr = _project_arc_length(points_latlon, centerline_latlon, chunk_size)

    threshold = lateral_threshold_for(empresaid)
    result = gps_e.with_columns([
        pl.Series("s", s_arr.astype(float), dtype=pl.Float64),
        pl.Series("lateral_m", lateral_arr.astype(float), dtype=pl.Float64),
    ]).filter(pl.col("lateral_m") <= threshold)

    return result


def _project_arc_length(
    points_latlon: np.ndarray,
    centerline_latlon: np.ndarray,
    chunk_size: int,
) -> tuple[np.ndarray, np.ndarray]:
    """Pure-numpy point-to-polyline projection using local flat-Earth coordinates.

    For each point, finds the closest centerline segment, projects orthogonally,
    and computes cumulative arc-length s (meters from polyline start) plus
    lateral offset (perpendicular distance in meters).

    Memory: O(chunk_size × n_segments) intermediate tensor. chunk_size=10_000
    with 50 segments ≈ 4 MB float32 — bounded regardless of total ping count.

    Source: build_notebook_03.py lines 396-427.
    """
    pts = np.asarray(points_latlon, dtype=float)
    cl = np.asarray(centerline_latlon, dtype=float)

    # Convert to meters (local flat-Earth at Arequipa latitude).
    pts_m = np.stack([pts[:, 0] * LAT_DEG_M, pts[:, 1] * LON_DEG_M], axis=1)
    cl_m = np.stack([cl[:, 0] * LAT_DEG_M, cl[:, 1] * LON_DEG_M], axis=1)

    seg_starts = cl_m[:-1]                              # (m-1, 2)
    seg_vecs = np.diff(cl_m, axis=0)                    # (m-1, 2)
    seg_norms_sq = (seg_vecs ** 2).sum(axis=1)          # (m-1,)
    seg_lengths = np.sqrt(seg_norms_sq)
    cum_s = np.concatenate([[0.0], np.cumsum(seg_lengths)])   # (m,)

    n = pts_m.shape[0]
    s_out = np.zeros(n, dtype=np.float32)
    lateral_out = np.zeros(n, dtype=np.float32)

    for start in range(0, n, chunk_size):
        end = min(start + chunk_size, n)
        chunk = pts_m[start:end]                                       # (c, 2)
        diff = chunk[:, None, :] - seg_starts[None, :, :]             # (c, m-1, 2)
        t = (diff * seg_vecs[None, :, :]).sum(axis=2) / seg_norms_sq[None, :]
        t = np.clip(t, 0.0, 1.0)                                       # (c, m-1)
        proj = seg_starts[None, :, :] + t[:, :, None] * seg_vecs[None, :, :]
        dist_sq = ((chunk[:, None, :] - proj) ** 2).sum(axis=2)       # (c, m-1)
        best_seg = dist_sq.argmin(axis=1)                              # (c,)
        best_t = np.take_along_axis(t, best_seg[:, None], axis=1).squeeze(1)
        s_out[start:end] = cum_s[best_seg] + best_t * seg_lengths[best_seg]
        lateral_out[start:end] = np.sqrt(
            np.take_along_axis(dist_sq, best_seg[:, None], axis=1).squeeze(1)
        )

    return s_out, lateral_out

## Module: direction

Inferencia de sentido ida/vuelta desde `sign(rolling_mean(ds, win=5))`.
El campo `direccion` se usa solo como verificación cruzada en E2.

In [ ]:
"""Direction inference from the sign of the smoothed arc-length derivative.

Primary method: sign(rolling_mean(ds, DIRECTION_SMOOTH_WIN)) per (empresaid, unidadid).
This is the SOLE primary source (decisiones-limpieza-fase2 §3.1). The `direccion`
heading field is used only as a cross-check diagnostic for empresas that have it
(E2/E4) and is never used to overwrite the primary signal.

Source: derived from build_notebook_03.py lines 471-504.
"""
from __future__ import annotations

import polars as pl



def infer_direction(gps: pl.DataFrame) -> pl.DataFrame:
    """Infer per-ping direction from sign(rolling_mean(ds, DIRECTION_SMOOTH_WIN)).

    Direction values:
      +1  = ida  (increasing s)
      -1  = vuelta (decreasing s)
       0  = undetermined (insufficient or ambiguous data at window start/end)

    Args:
        gps: must have (empresaid, unidadid, s) columns and be sorted by
             (empresaid, unidadid, time).

    Returns:
        gps + columns (ds_raw: Float64, ds_smooth: Float64, direction: Int8).

    Source: build_notebook_03.py lines 475-487.
    """
    win = PRODUCTIVE_PARAMS.direction_smooth_win

    gps = gps.with_columns([
        (
            pl.col("s") - pl.col("s").shift(1).over(["empresaid", "unidadid"])
        ).alias("ds_raw"),
    ])
    gps = gps.with_columns([
        pl.col("ds_raw")
          .rolling_mean(window_size=win, min_samples=1)
          .over(["empresaid", "unidadid"])
          .alias("ds_smooth"),
    ])
    gps = gps.with_columns([
        pl.when(pl.col("ds_smooth") > 0).then(pl.lit(1, dtype=pl.Int8))
          .when(pl.col("ds_smooth") < 0).then(pl.lit(-1, dtype=pl.Int8))
          .otherwise(pl.lit(0, dtype=pl.Int8))
          .alias("direction"),
    ])
    return gps


def cross_check_heading(gps: pl.DataFrame, empresaid: int) -> pl.DataFrame:
    """DIAGNOSTIC ONLY — add a heading_agrees column for empresas with GPS heading.

    For empresas with EMPRESA_CONFIG[e].has_heading = True, computes agreement
    between the primary direction signal and a threshold-based heading
    classification. Does NOT alter the primary `direction` column.

    - `direccion == 0` is treated as NULL (sentinel value, not north heading).
    - heading classified: 45–135° → +1 (ida), 225–315° → -1 (vuelta), else 0.
    - heading_agrees = (primary direction == heading direction) when both non-zero.

    For empresas without heading (has_heading=False, e.g. E59) this is a no-op
    that returns gps unchanged.

    Args:
        gps: frame with (empresaid, direction) columns.
        empresaid: empresa identifier.

    Returns:
        gps + column (heading_agrees: Boolean) when applicable; unchanged otherwise.
    """
    cfg = EMPRESA_CONFIG.get(empresaid)
    if cfg is None or not cfg.has_heading:
        return gps
    if "direccion" not in gps.columns:
        return gps

    gps = gps.with_columns([
        # Treat direccion == 0 as null (sentinel).
        pl.when(pl.col("direccion") == 0)
          .then(None)
          .otherwise(pl.col("direccion"))
          .alias("_heading_clean"),
    ])
    gps = gps.with_columns([
        pl.when(
            (pl.col("_heading_clean") >= 45) & (pl.col("_heading_clean") <= 135)
        ).then(pl.lit(1, dtype=pl.Int8))
          .when(
            (pl.col("_heading_clean") >= 225) & (pl.col("_heading_clean") <= 315)
        ).then(pl.lit(-1, dtype=pl.Int8))
          .otherwise(pl.lit(0, dtype=pl.Int8))
          .alias("_heading_dir"),
    ])
    gps = gps.with_columns([
        pl.when(
            (pl.col("direction") != 0) & (pl.col("_heading_dir") != 0)
        ).then(pl.col("direction") == pl.col("_heading_dir"))
          .otherwise(None)
          .alias("heading_agrees"),
    ]).drop(["_heading_clean", "_heading_dir"])

    return gps

## Module: trips

Segmentación de viajes (gap 30 min / reversal / terminal dwell 5 min)
y grilla de snapshots con alineación minuto-exacta (INV-6).

In [ ]:
"""Trip segmentation and snapshot grid construction.

assign_trip_ids — split each bus trajectory into trips on three cut conditions:
  1. GAP cut: dt_s > GAP_CUT_SECONDS (30 min) between consecutive pings.
  2. DIRECTION REVERSAL cut: primary direction flips +1↔-1 (transient 0s skipped).
  3. TERMINAL cut: bus stops near s_min/s_max for >= TERMINAL_DWELL_SECONDS (5 min).

build_snapshots — resample each bus to a minute-aligned uniform time grid, with
  linear interpolation of s and speed_kmh, and nearest-known direction by
  left-search.

Source: derived from build_notebook_03.py lines 606-660 (build_snapshots).
Trip segmentation is net-new for production — the probe deferred it.
"""
from __future__ import annotations

import numpy as np
import polars as pl



def _compute_trip_ids_for_bus(
    s_arr: np.ndarray,
    dt_s_arr: np.ndarray,
    speed_arr: np.ndarray,
    dir_arr: np.ndarray,
    time_arr: np.ndarray,
    s_min: float,
    s_max: float,
) -> np.ndarray:
    """Compute trip_id per ping for a single (empresaid, unidadid).

    Returns a uint32 array of the same length as the input arrays, where each
    element is the trip_id for that ping (monotonically non-decreasing).

    Cut conditions:
      1. GAP: dt_s > GAP_CUT_SECONDS
      2. REVERSAL: last-known direction flips +1 ↔ -1 (direction==0 skipped)
      3. TERMINAL EXIT: bus leaves a near-terminal stopped zone that lasted >= DWELL_SECONDS

    The terminal cut is placed on the EXIT ping (the first ping after leaving
    the dwell zone that exceeded the duration threshold).
    """
    n = len(s_arr)
    trip_ids = np.zeros(n, dtype=np.uint32)
    current_trip = np.uint32(0)

    # --- Pre-compute per-ping flags ---
    is_gap = np.zeros(n, dtype=bool)
    is_gap[1:] = dt_s_arr[1:] > GAP_CUT_SECONDS

    # REVERSAL: forward-fill direction ignoring 0s; detect sign flip.
    last_dir = 0
    prev_last_dir = 0
    is_reversal = np.zeros(n, dtype=bool)
    for i in range(n):
        d = int(dir_arr[i])
        if d != 0:
            if prev_last_dir != 0 and d != last_dir:
                # But we only cut on the non-zero flip, not on the first transition.
                # We set is_reversal at position i (the new direction starts here).
                is_reversal[i] = True
            prev_last_dir = last_dir
            last_dir = d

    # TERMINAL DWELL: track cumulative time near terminal while stopped.
    near = (s_arr < (s_min + TERMINAL_BAND_M)) | (s_arr > (s_max - TERMINAL_BAND_M))
    stopped = speed_arr < TERMINAL_MAX_SPEED_KMH
    in_dwell = near & stopped

    is_terminal_exit = np.zeros(n, dtype=bool)
    dwell_start_time = None
    dwell_block_exceeded = False

    for i in range(n):
        if in_dwell[i]:
            if dwell_start_time is None:
                dwell_start_time = time_arr[i]
                dwell_block_exceeded = False
            elapsed = float(time_arr[i] - dwell_start_time) / 1e9  # ns → s
            if elapsed >= TERMINAL_DWELL_SECONDS:
                dwell_block_exceeded = True
        else:
            if dwell_block_exceeded:
                # This is the EXIT ping.
                is_terminal_exit[i] = True
            dwell_start_time = None
            dwell_block_exceeded = False

    # --- Assemble trip_ids from cut flags ---
    for i in range(n):
        if i > 0 and (is_gap[i] or is_reversal[i] or is_terminal_exit[i]):
            current_trip += np.uint32(1)
        trip_ids[i] = current_trip

    return trip_ids


def assign_trip_ids(
    gps: pl.DataFrame,
    s_min: float | None = None,
    s_max: float | None = None,
) -> pl.DataFrame:
    """Assign a monotonic trip_id per (empresaid, unidadid) from three cut criteria.

    Cut conditions (any one triggers a new trip_id):
      1. GAP: dt_s > GAP_CUT_SECONDS between consecutive pings.
      2. REVERSAL: last-known direction flips +1 ↔ -1 (transient direction==0
         pings are skipped using forward-fill of the last non-zero direction).
      3. TERMINAL: bus is within TERMINAL_BAND_M of s_min or s_max AND stopped
         (speed_kmh < TERMINAL_MAX_SPEED_KMH) for >= TERMINAL_DWELL_SECONDS.
         The cut is placed on the EXIT ping of the dwell run (i.e. the first
         ping where the bus resumes movement or leaves the terminal band).

    trip_id is UInt32, monotonically increasing per bus, starting from 0 at
    the first ping of each (empresaid, unidadid). Trips of length < 2 pings
    are KEPT (downstream filters may drop them; we do not silently merge).

    Args:
        gps: must have (empresaid, unidadid, time, s, speed_kmh, direction, dt_s)
             sorted by (empresaid, unidadid, time).
        s_min: corridor start arc-length (meters). Computed from data if None.
        s_max: corridor end arc-length (meters). Computed from data if None.

    Returns:
        gps + column (trip_id: UInt32).

    Failure modes:
    - If terminal-cut boundary semantics flip (cut on ENTRY instead of EXIT),
      test_trips.py::test_terminal_cut_creates_new_trip_on_e59 catches it.
    - If reversal cut is placed on the direction==0 pings (short stops),
      trip count inflates; test_trips.py::test_gap_cut_creates_new_trip
      provides a stable baseline count.
    """
    if s_min is None:
        s_min = float(gps["s"].min() or 0.0)
    if s_max is None:
        s_max = float(gps["s"].max() or 0.0)

    gps = gps.sort(["empresaid", "unidadid", "time"])

    # Use row_index to guarantee correct positional mapping back to the sorted frame
    # after group_by (maintain_order=True guarantees group iteration order but not
    # row order within the full frame after re-join).
    gps_indexed = gps.with_row_index("_row_idx")
    trip_id_parts: list[pl.DataFrame] = []

    for keys, sub in gps_indexed.group_by(["empresaid", "unidadid"], maintain_order=True):
        sub_sorted = sub.sort("time")
        dt_s = sub_sorted["dt_s"].fill_null(0.0).to_numpy().astype(np.float64)
        s_arr = sub_sorted["s"].to_numpy().astype(np.float64)
        speed_arr = sub_sorted["speed_kmh"].fill_null(0.0).to_numpy().astype(np.float64)
        dir_arr = sub_sorted["direction"].to_numpy().astype(np.int64)
        time_arr = sub_sorted["time"].to_numpy().astype("datetime64[ns]").astype(np.int64)

        trip_ids = _compute_trip_ids_for_bus(
            s_arr, dt_s, speed_arr, dir_arr, time_arr, s_min, s_max
        )
        trip_id_parts.append(pl.DataFrame({
            "_row_idx": sub_sorted["_row_idx"],
            "trip_id": trip_ids,
        }))

    if not trip_id_parts:
        return gps.with_columns(pl.lit(0, dtype=pl.UInt32).alias("trip_id"))

    trip_id_df = pl.concat(trip_id_parts)
    result = gps_indexed.join(trip_id_df, on="_row_idx", how="left").drop("_row_idx")
    return result


def build_snapshots(
    gps: pl.DataFrame,
    grid_seconds: int = PRODUCTIVE_PARAMS.grid_seconds,
) -> pl.DataFrame:
    """Resample each bus to a minute-aligned uniform time grid per (empresaid, day).

    Grid alignment: uses epoch-floor pattern (t_min_s // grid_s) * grid_s to
    ensure all t_grid timestamps satisfy t.second == 0 (clarification #17 rule 1,
    INV-6).

    Interpolation:
      s         — linear interpolation (np.interp)
      speed_kmh — linear interpolation (null → 0.0 before interpolating)
      direction — nearest known by left-search (latest known state)
      trip_id   — nearest by left-search (when column is present)

    Only grid points within the bus's reported [t_min, t_max] window are emitted.
    Buses with < 2 pings are skipped.

    Source: build_notebook_03.py lines 606-660 with epoch-floor alignment added.
    """
    snaps_per_eday: list[pl.DataFrame] = []

    # Add day column if not present.
    if "day" not in gps.columns:
        gps = gps.with_columns(pl.col("time").dt.date().alias("day"))

    has_trip_id = "trip_id" in gps.columns
    has_lateral_m = "lateral_m" in gps.columns

    for keys, sub_eday in gps.group_by(["empresaid", "day"], maintain_order=True):
        e, day = keys[0], keys[1]

        # Compute minute-aligned epoch-floor grid (INV-6 / clarification #17 rule 1).
        # Use numpy int64 microseconds (matching polars Datetime["us"] storage) to
        # avoid Python datetime.timestamp() UTC/local ambiguity.
        t_min_us = int(sub_eday["time"].to_numpy().astype("datetime64[us]").astype(np.int64).min())
        t_max_us = int(sub_eday["time"].to_numpy().astype("datetime64[us]").astype(np.int64).max())
        grid_us = grid_seconds * 1_000_000   # grid in microseconds
        t_grid_us = np.arange(
            (t_min_us // grid_us) * grid_us,
            ((t_max_us // grid_us) + 1) * grid_us + 1,
            grid_us,
            dtype=np.int64,
        )
        # Also keep ns for interp (t_arr will be ns from the per-bus conversion below).
        t_grid_ns = t_grid_us * 1_000

        for bus_keys, sub in sub_eday.group_by(["unidadid"], maintain_order=True):
            bus = bus_keys[0]
            sub_sorted = sub.sort("time")
            t_arr = sub_sorted["time"].to_numpy().astype("datetime64[us]").astype(np.int64) * 1_000
            s_arr = sub_sorted["s"].to_numpy().astype(np.float64)
            v_arr = sub_sorted["speed_kmh"].fill_null(0.0).to_numpy().astype(np.float64)
            d_arr = sub_sorted["direction"].to_numpy().astype(np.int64)

            if len(t_arr) < 2:
                continue

            # Only interpolate within the bus's reported window.
            in_window = (t_grid_ns >= t_arr[0]) & (t_grid_ns <= t_arr[-1])
            if not in_window.any():
                continue

            tg = t_grid_ns[in_window]
            s_interp = np.interp(tg, t_arr, s_arr)
            v_interp = np.interp(tg, t_arr, v_arr)

            # Direction: nearest known (left-search), carrying the latest known state.
            idx_left = np.searchsorted(t_arr, tg, side="right") - 1
            idx_left = np.clip(idx_left, 0, len(d_arr) - 1)
            d_interp = d_arr[idx_left]

            row_data: dict = {
                "empresaid": np.full(len(tg), int(e), dtype=np.int64),
                "day": [day] * len(tg),
                "t": tg,
                "unidadid": np.full(len(tg), int(bus), dtype=np.int64),
                "s": s_interp.astype(np.float64),
                "speed_kmh": v_interp.astype(np.float64),
                "direction": d_interp.astype(np.int8),
            }

            if has_trip_id:
                tid_arr = sub_sorted["trip_id"].to_numpy().astype(np.uint32)
                idx_trip = np.searchsorted(t_arr, tg, side="right") - 1
                idx_trip = np.clip(idx_trip, 0, len(tid_arr) - 1)
                row_data["trip_id"] = tid_arr[idx_trip]

            if has_lateral_m:
                # Linear interpolation of lateral_m alongside s/speed_kmh.
                # lateral_m is a continuous geometric quantity (orthogonal distance
                # to centerline) — same regularity class as s. np.interp handles
                # null/NaN by propagating them; fill_null(0.0) is intentionally NOT
                # used here because a null lateral_m carries meaning (ping without
                # projection), and we want to propagate it faithfully.
                lat_arr = sub_sorted["lateral_m"].fill_null(float("nan")).to_numpy().astype(np.float64)
                lat_interp = np.interp(tg, t_arr, lat_arr)
                # Convert NaN back to null via a float64 series.
                lat_series = pl.Series("lateral_m", lat_interp, dtype=pl.Float64)
                row_data["lateral_m"] = lat_interp

            snaps_per_eday.append(pl.DataFrame(row_data))

    if not snaps_per_eday:
        return pl.DataFrame()

    snaps = pl.concat(snaps_per_eday)
    # t is stored as int64 nanoseconds (from t_grid_ns); convert to Datetime[us].
    snaps = snaps.with_columns(
        (pl.col("t") // 1_000).cast(pl.Datetime("us")).alias("t")
    )
    return snaps

## Module: headways

C.2 trailing crossing — pure polars+numpy. Para pares sin historial previo
se emite `delta_t_min = null` (NO se descarta — clarification #17 rule 2).
Winsorización aplica en Fase 5, NO aquí (Caveat 2).

In [ ]:
"""Headway computation via C.2 — trailing crossing (pure polars + numpy).

compute_pairs — build the pair structure: for each (empresaid, day, t, direction),
    sort buses by s and emit one (front, back) row per consecutive pair.

compute_headways_c2 — for each pair, find the most recent past time when bus_back
    crossed s_front in its trajectory, and compute delta_t_min = T - t_cross.

Clarification #17 rule 2: when no crossing is found, the row is EMITTED with
delta_t_min = null (NOT dropped). This preserves pair_rank density (INV-3) and
n_buses consistency.

Note on NULL rows: they appear mostly in the first GRID_SECONDS of a bus's
trajectory (before bus_back has driven through any front position). The NULL
fraction should be < 5% globally; if higher, investigate trip-segmentation edge
cases. (Caveat per clarification #17 §Frequency expectation.)

Source: rewrite of build_notebook_03.py lines 751-813. The probe used pandas
    conversion + row-level Python loop. This implementation uses a trajectory
    index built with polars group_by + numpy numpy-escape per back-bus group
    (O(K) per group, not per pair).

winsorization: delta_t_min is stored RAW. Winsorization is a Fase 5 transformation
    applied at training time, NOT here (decisiones-headway-fase2.md §4 Caveat 2).
"""
from __future__ import annotations

import numpy as np
import polars as pl



def compute_pairs(snapshots: pl.DataFrame) -> pl.DataFrame:
    """Build consecutive (front, back) bus pairs per (empresaid, day, t, direction).

    For each snapshot group sorted by s (ascending), bus at rank i is "front" and
    bus at rank i-1 is "back". Drops direction == 0 rows.

    Lateral pair filter (R-LAT3): after pair formation, drops pairs where
    |lateral_m_front − lateral_m_back| > lateral_pair_threshold_for(empresaid).
    Rows where either lateral value is null are RETAINED (conservative).
    Filter is applied only when the input snapshot frame contains a `lateral_m`
    column. When the column is absent, all pairs are retained (backward-compatible).

    Args:
        snapshots: output of trips.build_snapshots with columns
                   (empresaid, day, t, unidadid, s, speed_kmh, direction[, lateral_m]).

    Returns:
        pl.DataFrame with columns:
          empresaid, day, t, direction,
          pair_rank (Int32, 1-indexed, dense per group),
          bus_front (Int64), bus_back (Int64),
          s_front (Float64), s_back (Float64),
          speed_front_kmh (Float64), speed_back_kmh (Float64),
          n_buses (Int32)[, lateral_m_front (Float64), lateral_m_back (Float64)].
          The lateral columns are present only when the input has lateral_m.

    Failure mode: if shift(1) is applied before sort, pair assignment is wrong;
    test_headways.py::test_pair_structure_count catches this.
    """
    has_lateral = "lateral_m" in snapshots.columns

    s = snapshots.filter(pl.col("direction") != 0)
    s = s.sort(["empresaid", "day", "t", "direction", "s"])

    group_cols = ["empresaid", "day", "t", "direction"]

    shift_exprs = [
        pl.col("s").shift(1).over(group_cols).alias("s_back"),
        pl.col("unidadid").shift(1).over(group_cols).alias("bus_back"),
        pl.col("speed_kmh").shift(1).over(group_cols).alias("speed_back_kmh"),
        pl.col("unidadid").count().over(group_cols).cast(pl.Int32).alias("n_buses"),
        # cum_count starts at 1 for the first row; after dropping the first row
        # (the "back" reference is null) we get ranks 2..N. Subtract 1 to get 1..N-1.
        (pl.col("s").cum_count().over(group_cols).cast(pl.Int32) - 1).alias("pair_rank"),
    ]
    if has_lateral:
        # Shift lateral_m to get the back-bus value after pairing.
        shift_exprs.append(
            pl.col("lateral_m").shift(1).over(group_cols).alias("lateral_m_back_raw")
        )
        # The front bus keeps its own lateral_m (renamed after select).
        shift_exprs.append(
            pl.col("lateral_m").alias("lateral_m_front_raw")
        )

    s = s.with_columns(shift_exprs)

    # Drop the first bus in each group (shift produces null for it).
    s = s.filter(pl.col("s_back").is_not_null())

    if has_lateral:
        # Step 6: filter cross-street pairs.
        # Build per-empresa threshold mapping via Python-side lookup (task note:
        # fallback from vectorised when/then if empresa list varies).
        empresa_ids = s["empresaid"].unique().to_list()
        threshold_map = {int(e): lateral_pair_threshold_for(int(e)) for e in empresa_ids}

        # Build a Polars expression: pl.col("empresaid").replace(mapping, default=global)
        # Conservative rule: retain if either lateral value is null.
        # retain when: lateral_m_front_raw IS NULL
        #           OR lateral_m_back_raw IS NULL
        #           OR abs(front - back) <= threshold
        global_threshold = PRODUCTIVE_PARAMS.lateral_pair_threshold_m
        keep_expr = (
            pl.col("lateral_m_front_raw").is_null()
            | pl.col("lateral_m_back_raw").is_null()
            | (
                (pl.col("lateral_m_front_raw") - pl.col("lateral_m_back_raw")).abs()
                <= pl.col("empresaid").replace_strict(
                    threshold_map,
                    default=global_threshold,
                    return_dtype=pl.Float64,
                )
            )
        )
        s = s.filter(keep_expr)

    select_exprs = [
        "empresaid",
        "day",
        "t",
        "direction",
        "pair_rank",
        pl.col("unidadid").alias("bus_front"),
        pl.col("bus_back").cast(pl.Int64),
        pl.col("s").alias("s_front"),
        pl.col("s_back").cast(pl.Float64),
        pl.col("speed_kmh").alias("speed_front_kmh"),
        pl.col("speed_back_kmh").cast(pl.Float64),
        "n_buses",
    ]
    if has_lateral:
        select_exprs += [
            pl.col("lateral_m_front_raw").cast(pl.Float64).alias("lateral_m_front"),
            pl.col("lateral_m_back_raw").cast(pl.Float64).alias("lateral_m_back"),
        ]

    return s.select(select_exprs)


def _find_last_crossing_ns(
    t_arr: np.ndarray,
    s_arr: np.ndarray,
    T_ns: int,
    s_front: float,
    max_lookback_ns: float | None = None,
) -> float | None:
    """Find the most recent time (nanoseconds) when bus_back's s crossed s_front.

    Uses the probe's sign-change scan (build_notebook_03.py lines 796-806) on the
    trajectory of bus_back restricted to t <= T. Linear interpolation over the
    bracket gives the exact crossing nanosecond.

    Args:
        t_arr: int64 nanosecond timestamps, sorted ascending.
        s_arr: float64 arc-length values at those timestamps.
        T_ns:  snapshot time in nanoseconds (restrict to t <= T).
        s_front: arc-length of the front bus at T.
        max_lookback_ns: when not None, crossings older than this many nanoseconds
            before T are treated as 'no crossing found' and return None. Prevents
            stale historical crossings in multi-filar corridors (e.g. E2 Arequipa)
            from being emitted as absurd delta_t_min values (decisiones-headway-fase2 §3).

    Returns:
        t_cross in nanoseconds (float), or None if no crossing exists or the
        crossing is older than max_lookback_ns.
    """
    cutoff = int(np.searchsorted(t_arr, T_ns, side="right"))
    if cutoff < 2:
        return None

    s_past = s_arr[:cutoff]
    t_past = t_arr[:cutoff]

    diff = s_past - s_front

    # Case 1: exact zero crossing — bus_back was exactly at s_front.
    zero_mask = diff == 0.0
    if zero_mask.any():
        i = int(np.where(zero_mask)[0][-1])
        t_cross = float(t_past[i])
        if max_lookback_ns is not None and (T_ns - t_cross) > max_lookback_ns:
            return None
        return t_cross

    # Case 2: sign-change crossing — bus_back's s straddled s_front.
    signs = np.sign(diff)
    cross_mask = (signs[:-1] * signs[1:]) < 0

    if not cross_mask.any():
        return None

    # Most recent crossing (last True in cross_mask).
    i = int(np.where(cross_mask)[0][-1])

    ds = s_past[i + 1] - s_past[i]
    if ds == 0.0:
        return None

    frac = float((s_front - s_past[i]) / ds)
    t_cross = float(t_past[i]) + frac * float(t_past[i + 1] - t_past[i])
    if max_lookback_ns is not None and (T_ns - t_cross) > max_lookback_ns:
        return None
    return t_cross


def compute_headways_c2(
    snapshots: pl.DataFrame,
    gps: pl.DataFrame,
    min_buses: int = PRODUCTIVE_PARAMS.min_buses_per_snapshot,
    max_lookback_minutes: float = PRODUCTIVE_PARAMS.max_interpolation_lookback_minutes,
) -> pl.DataFrame:
    """C.2 trailing-crossing headway (pure polars + numpy).

    For each pair (bus_front at s_front, bus_back) at snapshot time T, finds the
    most recent past time when bus_back's s-trajectory crossed s_front (in the
    same direction) and computes:

        delta_t_min = (T - t_cross).total_seconds() / 60

    When no crossing is found (e.g. bus_back just entered the corridor and has
    not yet crossed s_front): emits the row with delta_t_min = null (NOT dropped).
    This is clarification #17 rule 2 — preserves INV-3 (dense pair_rank) and
    INV-4 (n_buses consistent with active bus count).

    Crossings whose interpolated t_cross is older than max_lookback_minutes are
    treated as 'no crossing' and emitted with delta_t_min = NULL (same semantics
    as clarification §2). This bound exists because multi-filar corridors project
    unrelated buses to the same s axis; without it, np.searchsorted finds ancient
    crossings and emits absurd headways (e.g. ~112 days on E2 dir=1).

    Algorithm:
    1. Build a trajectory index: group gps by (empresaid, unidadid, direction)
       → (t_arr, s_arr) sorted by time. This is O(N) per group.
    2. Build the pair frame via compute_pairs.
    3. Iterate groups (empresaid, bus_back, direction): for all pairs in this
       group, run the numpy sign-change scan and record delta_t_min. This is
       O(P_k × K_k) per group where P_k = pairs for this back-bus and K_k = traj
       length. Reassemble via an explicit row-index join.

    Args:
        snapshots: output of trips.build_snapshots.
        gps: post-projection, post-direction frame (full trajectory for crossing
             lookup). Should be filtered to the relevant empresa and day range.
        min_buses: drop snapshot groups with fewer buses (INV-4).
        max_lookback_minutes: crossings older than this many minutes before T are
            emitted as NULL (same as no-crossing). Default from ProductiveParams.

    Returns:
        pl.DataFrame matching R7 schema:
          t, direction, pair_rank (Int32), bus_front (Int64), bus_back (Int64),
          s_front, s_back, speed_front_kmh, speed_back_kmh,
          delta_t_min (Float64, may be null per clarification #17 rule 2),
          n_buses (Int32).

    Failure mode: if the pandas-conversion path is accidentally reintroduced,
    performance collapses on 47M-row E2 data. test_headways.py guards the polars
    purity requirement.
    """
    pairs = compute_pairs(snapshots)

    # Drop pairs from too-small snapshots (INV-4: n_buses >= min_buses).
    pairs = pairs.filter(pl.col("n_buses") >= min_buses)
    if pairs.is_empty():
        return pairs.with_columns(pl.lit(None, dtype=pl.Float64).alias("delta_t_min"))

    # Convert minutes → nanoseconds ONCE (kernel works in nanoseconds throughout).
    max_lookback_ns = float(max_lookback_minutes) * 60.0 * 1e9

    # --- Build trajectory index ---
    # Group gps by (empresaid, unidadid, direction) → sorted (t_arr, s_arr).
    gps_dir = gps.filter(pl.col("direction") != 0)
    traj_index: dict[tuple[int, int, int], tuple[np.ndarray, np.ndarray]] = {}

    for keys, sub in gps_dir.group_by(
        ["empresaid", "unidadid", "direction"], maintain_order=False
    ):
        e, bus, dirc = int(keys[0]), int(keys[1]), int(keys[2])
        # Use microsecond-based int64 (Datetime["us"]) × 1000 → nanoseconds.
        t_arr = sub["time"].to_numpy().astype("datetime64[us]").astype(np.int64) * 1_000
        s_arr = sub["s"].to_numpy().astype(np.float64)
        order = np.argsort(t_arr)
        traj_index[(e, bus, dirc)] = (t_arr[order], s_arr[order])

    # --- Compute delta_t_min per pair ---
    # Attach a row index to pairs for result reassembly.
    pairs_indexed = pairs.with_row_index("_row_idx")

    # t column: snapshots use Datetime["us"], convert to nanoseconds for the lookup.
    t_ns_all = (
        pairs_indexed["t"].to_numpy().astype("datetime64[us]").astype(np.int64) * 1_000
    )
    s_front_all = pairs_indexed["s_front"].to_numpy().astype(np.float64)
    e_all = pairs_indexed["empresaid"].to_numpy().astype(np.int64)
    bus_back_all = pairs_indexed["bus_back"].to_numpy().astype(np.int64)
    dir_all = pairs_indexed["direction"].to_numpy().astype(np.int64)
    row_idx_all = pairs_indexed["_row_idx"].to_numpy().astype(np.int64)

    n = len(pairs_indexed)
    delta_t_min = np.full(n, np.nan, dtype=np.float64)

    # Iterate per (empresaid, bus_back, direction) group — O(P_k × K_k) per group.
    for keys, sub_idx in pairs_indexed.group_by(
        ["empresaid", "bus_back", "direction"], maintain_order=False
    ):
        e, bus, dirc = int(keys[0]), int(keys[1]), int(keys[2])
        traj_key = (e, bus, dirc)
        if traj_key not in traj_index:
            continue

        t_arr, s_arr = traj_index[traj_key]

        row_indices = sub_idx["_row_idx"].to_numpy().astype(np.int64)
        T_ns_group = t_ns_all[row_indices]
        s_front_group = s_front_all[row_indices]

        for j, (T_ns, sf) in enumerate(zip(T_ns_group, s_front_group)):
            t_cross = _find_last_crossing_ns(
                t_arr, s_arr, int(T_ns), float(sf),
                max_lookback_ns=max_lookback_ns,
            )
            if t_cross is not None:
                dt_ns = float(T_ns) - t_cross
                delta_t_min[row_indices[j]] = dt_ns / 1e9 / 60.0

    # Reassemble: NaN → null (clarification #17 rule 2 — emit null NOT drop).
    delta_series = pl.Series("delta_t_min", delta_t_min, dtype=pl.Float64)
    delta_series = delta_series.set(delta_series.is_nan(), None)

    result = pairs_indexed.drop("_row_idx").with_columns(delta_series)

    # Final schema cleanup: select R7 columns, preserving lateral diagnostic
    # columns when the upstream compute_pairs emitted them (R-LAT4 / AC-S1 / AC-S2).
    r7_cols = [
        "t",
        "direction",
        "pair_rank",
        "bus_front",
        "bus_back",
        "s_front",
        "s_back",
        "speed_front_kmh",
        "speed_back_kmh",
        "delta_t_min",
        "n_buses",
    ]
    if "lateral_m_front" in pairs_indexed.columns:
        r7_cols.append("lateral_m_front")
    if "lateral_m_back" in pairs_indexed.columns:
        r7_cols.append("lateral_m_back")
    return result.select(r7_cols)


def filter_snapshot_size(headways: pl.DataFrame, min_buses: int) -> pl.DataFrame:
    """Drop rows belonging to snapshots with fewer than min_buses active buses.

    INV-4: n_buses >= min_buses for all rows.

    Implementation: filter on the pre-computed n_buses column (set by compute_pairs).
    """
    return headways.filter(pl.col("n_buses") >= min_buses)

## Ejecutar pipeline por empresa

Carga `clean_gps.parquet`, aplica todos los módulos en orden de dependencia,
y escribe los artefactos intermedios por corredor.

In [ ]:

lf = (
    pl.scan_parquet(INPUT)
    .filter(
        pl.col("empresaid").is_in(EMPRESAS)
        & pl.col("time").is_not_null()
        & pl.col("lat").is_not_null() & pl.col("lon").is_not_null()
        & (pl.col("lat") != 0) & (pl.col("lon") != 0)
    )
    .with_columns(pl.col("time").dt.date().alias("day"))
    .sort(["empresaid", "unidadid", "time"])
)
gps_all = lf.collect(engine="streaming")
print(f"Rows loaded: {gps_all.height:,}")

for empresaid in EMPRESAS:
    print(f"\n--- Empresa {empresaid} ---")
    sub = gps_all.filter(pl.col("empresaid") == empresaid)

    sub = attach_observed_speed(sub)
    centerline = build_centerline(sub, empresaid=empresaid)
    sub = project_to_centerline(sub, centerline, empresaid=empresaid)
    sub = infer_direction(sub)
    sub = assign_trip_ids(sub)
    snaps = build_snapshots(sub)
    heads = compute_headways_c2(snaps, sub)

    out_gps = OUTPUT_DIR / f"cleaned_gps_E{empresaid}.parquet"
    out_hw = OUTPUT_DIR / f"headways_E{empresaid}.parquet"
    sub.rename({"time": "t"}).select(
        ["unidadid", "t", "lat", "lon", "s", "direction", "speed_kmh", "lateral_m"]
    ).write_parquet(out_gps)
    heads.write_parquet(out_hw)

    print(f"  cleaned_gps:  {sub.height:,} rows → {out_gps}")
    print(f"  headways:     {heads.height:,} rows → {out_hw}")
    print(f"  non-null hw:  {heads.filter(pl.col('delta_t_min').is_not_null()).height:,}")

## Auditoría de sanidad

Verifica los invariantes del spec (INV-1..INV-8) sobre los parquets producidos.
R7 schema (v4): headways parquet contiene `lateral_m_front` y `lateral_m_back`
como columnas diagnósticas adicionales (multi-filar-disambiguation). Ver
`docs/decisiones-headway-fase2.md §3` para el threshold y el protocolo de
calibración en notebook 04b Figura 7.

In [ ]:

for empresaid in EMPRESAS:
    out_gps = OUTPUT_DIR / f"cleaned_gps_E{empresaid}.parquet"
    out_hw = OUTPUT_DIR / f"headways_E{empresaid}.parquet"
    if not out_gps.exists() or not out_hw.exists():
        print(f"E{empresaid}: output files not found, skip audit")
        continue

    gps_e = pl.read_parquet(out_gps)
    hw_e = pl.read_parquet(out_hw)

    print(f"\n=== E{empresaid} audit ===")
    print(f"  cleaned_gps: {gps_e.height:,} rows, {gps_e.width} cols")
    print(f"  headways:    {hw_e.height:,} rows, {hw_e.width} cols")

    # INV-6: all t values have second == 0
    if hw_e.height > 0:
        bad_seconds = hw_e.filter(pl.col("t").dt.second() != 0).height
        print(f"  INV-6 violations (t.second != 0): {bad_seconds}")

    # INV-4: n_buses >= 2
    if hw_e.height > 0:
        bad_n = hw_e.filter(pl.col("n_buses") < 2).height
        print(f"  INV-4 violations (n_buses < 2): {bad_n}")

    # INV-7: bus_front != bus_back
    if hw_e.height > 0:
        bad_pair = hw_e.filter(pl.col("bus_front") == pl.col("bus_back")).height
        print(f"  INV-7 violations (bus_front == bus_back): {bad_pair}")

    # INV-8: lateral_m <= 300
    if gps_e.height > 0:
        bad_lat = gps_e.filter(pl.col("lateral_m") > 300.0).height
        print(f"  INV-8 violations (lateral_m > 300): {bad_lat}")

    # NULL rate in delta_t_min
    if hw_e.height > 0:
        null_frac = hw_e.filter(pl.col("delta_t_min").is_null()).height / hw_e.height
        print(f"  delta_t_min null fraction: {null_frac:.1%}")
        print(f"  delta_t_min stats: {hw_e['delta_t_min'].drop_nulls().describe()}")

    # n_pairs_efectivo per day (derive 'day' from 't' — headways schema has no 'day' column)
    if hw_e.height > 0:
        pairs_per_day = (
            hw_e.filter(pl.col("delta_t_min").is_not_null())
            .with_columns(pl.col("t").dt.date().alias("day"))
            .group_by("day").len().sort("day")
        )
        print(f"  pairs_efectivo/day: min={pairs_per_day['len'].min():,} "
              f"max={pairs_per_day['len'].max():,} mean={int(pairs_per_day['len'].mean()):,}")